# Den 4: Zpracování textových dat – Cvičení 2 (Lemmatizace)

## Zadání úkolu:
1. Načtěte data ze souboru `imdb_reviews_preprocessed_1.csv` do proměnné `imdb_reviews` pomocí knihovny **Pandas**.
2. Napište funkci, která jako argument přijme vyčištěný text recenze (`review_cleaned`) a vrátí text skládající se ze slov ve tvaru **lemmat**.
3. Do DataFrame přidejte sloupec `review_lemmatized`, který vznikne aplikací této funkce na sloupec `review_cleaned` pomocí metody `.apply()`.
4. Uložte výsledný DataFrame do souboru `.csv` jako `imdb_reviews_preprocessed_2.csv`.

### Krok 1: Import knihoven a načtení dat
Načteme data vyčištěná v prvním cvičení (odstraněná interpunkce, převedeno na malá písmena, vyjmuta stop-slova).

In [ ]:
from pathlib import Path
import pandas as pd
import spacy

# Cesta k souboru (kompatibilní s rootem i podsložkou)
data_path = Path("../data/imdb_reviews_preprocessed_1.csv")
if not data_path.exists():
    data_path = Path("data/imdb_reviews_preprocessed_1.csv")

imdb_reviews = pd.read_csv(data_path)
print(f"Načteno recenzí: {len(imdb_reviews)}")
imdb_reviews.head()

### Krok 2: Definice lemmatizační funkce
Využijeme NLP model `en_core_web_sm` z knihovny `spaCy`. Aby byl běh rychlý, vypneme syntaktický parser a rozpoznávání pojmenovaných entit (NER), které pro lemmatizaci nepotřebujeme.

In [ ]:
# Inicializace modelu spaCy
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatize_text(text: str) -> str:
    """
    Převede zadaný text na řetězec lemmatizovaných tokenů.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc if not token.is_space])

# Otestování funkce na krátkém textu
sample_input = "the striped wolves were studying the running feet better than universities"
print("Vstup: ", sample_input)
print("Lemmata:", lemmatize_text(sample_input))

### Krok 3: Aplikace lemmatizace na celý dataset pomocí `.apply()`

In [ ]:
print("Spouštím lemmatizaci recenzí přes .apply()...")
imdb_reviews["review_lemmatized"] = imdb_reviews["review_cleaned"].apply(lemmatize_text)
print("Lemmatizace dokončena!")
imdb_reviews[["sentiment", "review_cleaned", "review_lemmatized"]].head()

### Krok 4: Uložení do souboru `imdb_reviews_preprocessed_2.csv`

In [ ]:
output_path = Path("../data/imdb_reviews_preprocessed_2.csv")
if not output_path.parent.exists():
    output_path = Path("data/imdb_reviews_preprocessed_2.csv")

imdb_reviews.to_csv(output_path, index=False)
print(f"DataFrame úspěšně uložen do: {output_path.resolve()}")

### Krok 5: Analýza slovníku a vlivu lemmatizace
Ověříme, o kolik procent se zredukoval prostor unikátních příznaků (velikost slovníku) a jaké transformace byly nejčastější.

In [ ]:
from collections import Counter

cleaned_words = [w for text in imdb_reviews["review_cleaned"].dropna() for w in text.split()]
lemmatized_words = [w for text in imdb_reviews["review_lemmatized"].dropna() for w in text.split()]

vocab_cleaned = set(cleaned_words)
vocab_lemmatized = set(lemmatized_words)

print(f"Celkem tokenů: {len(cleaned_words):,}")
print(f"Unikátních slov před lemmatizací: {len(vocab_cleaned):,}")
print(f"Unikátních lemmat po lemmatizaci: {len(vocab_lemmatized):,}")
reduction = (1 - len(vocab_lemmatized) / len(vocab_cleaned)) * 100
print(f"Redukce slovníku (dimensionality reduction): {reduction:.2f} %")